# GitHub Wrapped — GDS Notebook
## Graph Projection & Louvain Community Detection

This notebook:
1. Connects to AuraDB using environment-variable credentials
2. Projects a **person-to-person collaboration graph** (persons who co-appear on the same PR as author + reviewer)
3. Runs **Louvain community detection** on the projected graph
4. Writes **community IDs back** to `Person.community` in AuraDB
5. Prints community count and size distribution

**Prerequisites**
- AuraDB instance with data imported (tasks 001–003 complete)
- GDS plugin enabled on the AuraDB instance (required for `graphdatascience` client)
- Environment variables set: `NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD`

Install dependencies with:
```bash
uv sync --extra notebook
```

## 1. Setup & Connection

In [ ]:
import os
import pathlib

def _load_env_file(path: pathlib.Path) -> None:
    """Minimal .env parser — sets os.environ for KEY=VALUE lines (no override)."""
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, _, val = line.partition('=')
        key = key.strip()
        val = val.strip().strip('"').strip("'")
        if key and key not in os.environ:  # don't override existing env vars
            os.environ[key] = val

# Search for integration.env: current dir, then parent (notebooks/ → project root)
_candidates = [
    pathlib.Path.cwd() / 'integration.env',
    pathlib.Path.cwd().parent / 'integration.env',
]
_loaded = False
for _p in _candidates:
    if _p.exists():
        _load_env_file(_p)
        print(f'Loaded env from: {_p}')
        _loaded = True
        break

if not _loaded:
    print('No integration.env found — using system environment variables')

In [ ]:
from graphdatascience import GraphDataScience

NEO4J_URI      = os.environ['NEO4J_URI']
NEO4J_USERNAME = os.environ['NEO4J_USERNAME']
NEO4J_PASSWORD = os.environ['NEO4J_PASSWORD']
NEO4J_DATABASE = os.environ.get('NEO4J_DATABASE', 'neo4j')

print(f'Connecting to: {NEO4J_URI}')
print(f'Database:      {NEO4J_DATABASE}')

gds = GraphDataScience(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD),
    database=NEO4J_DATABASE,
)

print(f'GDS version:   {gds.version()}')

## 2. Explore Existing Data

In [ ]:
# Count persons, PRs, AUTHORED and REVIEWED relationships
counts = gds.run_cypher("""
MATCH (p:Person) WITH count(p) AS persons
MATCH (pr:PullRequest) WITH persons, count(pr) AS prs
MATCH ()-[a:AUTHORED]->() WITH persons, prs, count(a) AS authored
MATCH ()-[r:REVIEWED]->() WITH persons, prs, authored, count(r) AS reviewed
RETURN persons, prs, authored, reviewed
""")
print('Data summary:')
print(counts.to_string(index=False))

## 3. Build Collaboration Edge List (Cypher)

The collaboration graph is **person ↔ person** (undirected): two persons are connected if one **authored** and the other **reviewed** the same PR.

We use a Cypher query to compute the co-occurrence count as the edge weight.

In [ ]:
# Preview the top collaboration pairs (author → reviewer, weight = shared PRs)
sample = gds.run_cypher("""
MATCH (author:Person)-[:AUTHORED]->(pr:PullRequest)<-[:REVIEWED]-(reviewer:Person)
WHERE author <> reviewer
WITH author, reviewer, count(DISTINCT pr) AS sharedPRs
ORDER BY sharedPRs DESC
LIMIT 15
RETURN author.login AS author, reviewer.login AS reviewer, sharedPRs
""")
print('Top collaboration pairs (author, reviewer, shared PRs):')
sample

## 4. GDS Graph Projection

We project a **native undirected Person–Person** graph using a Cypher aggregation projection. Each edge weight is the number of PRs where one person authored and the other reviewed.

In [ ]:
GRAPH_NAME = 'collab-graph'

# Drop existing projection if present (makes notebook idempotent)
exists_result = gds.graph.exists(GRAPH_NAME)
if exists_result['exists']:
    gds.graph.drop(GRAPH_NAME)
    print(f'Dropped existing projection: {GRAPH_NAME}')

# Project person-to-person collaboration graph via Cypher aggregation projection.
# The query must end with RETURN gds.graph.project(...).
# Each node is a Person; edge weight = number of shared PRs (author ↔ reviewer on same PR).
g, result = gds.graph.cypher.project(
    """
    MATCH (author:Person)-[:AUTHORED]->(pr:PullRequest)<-[:REVIEWED]-(reviewer:Person)
    WHERE id(author) < id(reviewer)
    WITH author, reviewer, count(DISTINCT pr) AS sharedPRs
    RETURN gds.graph.project($graphName, author, reviewer,
        { relationshipType: 'COLLABORATED',
          relationshipProperties: { weight: sharedPRs }
        },
        { undirectedRelationshipTypes: ['COLLABORATED'] }
    )
    """,
    database=NEO4J_DATABASE,
    graphName=GRAPH_NAME
)

print(f'Graph projected: {GRAPH_NAME}')
print(f'  Nodes:         {g.node_count()}')
print(f'  Relationships: {g.relationship_count()}')
print(f'  Memory:        {result["memoryUsage"]}')

## 5. Louvain Community Detection

In [ ]:
# Run Louvain on the projected graph
# gds.louvain.mutate stores community IDs in the in-memory graph (mutateProperty)
# relationshipWeightProperty='weight' uses the sharedPRs count as edge weight
louvain_result = gds.louvain.mutate(
    g,
    mutateProperty='community',
    relationshipWeightProperty='weight',
    includeIntermediateCommunities=False,
)

print('Louvain result:')
print(f'  Communities found:  {louvain_result["communityCount"]}')
print(f'  Modularity:         {louvain_result["modularity"]:.4f}')
print(f'  Iterations:         {louvain_result["ranIterations"]}')
print(f'  Nodes classified:   {louvain_result["nodePropertiesWritten"]}')

In [ ]:
import pandas as pd

# Stream community assignments from the projected graph to inspect distribution
# gds.louvain.stream re-runs Louvain and returns nodeId + communityId per row
community_df = gds.louvain.stream(
    g,
    relationshipWeightProperty='weight',
)

print(f'Total persons with community: {len(community_df)}')
community_df.head(10)

In [ ]:
# Community size distribution
size_dist = community_df.groupby('communityId').size().reset_index(name='size')
size_dist = size_dist.sort_values('size', ascending=False).reset_index(drop=True)

print(f'Number of communities:   {len(size_dist)}')
print(f'Largest community:       {size_dist["size"].max()} persons')
print(f'Smallest community:      {size_dist["size"].min()} persons')
print(f'Median community size:   {size_dist["size"].median()}')
print()
print('Top 10 largest communities:')
size_dist.head(10)

In [ ]:
# Size distribution histogram (text-based)
bins = [1, 2, 5, 10, 20, 50, 100, 200]
labels = ['1', '2–4', '5–9', '10–19', '20–49', '50–99', '100–199', '200+']
cuts = pd.cut(
    size_dist['size'],
    bins=[0, 1, 4, 9, 19, 49, 99, 199, 10000],
    labels=labels
)
hist = cuts.value_counts().sort_index()
print('Community size distribution:')
for label, cnt in hist.items():
    bar = '█' * min(cnt, 40)
    print(f'  {label:>8} members: {bar} ({cnt})')

## 6. Write Community IDs Back to AuraDB

We write the `community` property from the in-memory projected graph back to the live `Person` nodes in AuraDB.

In [ ]:
# Write community property from projected graph back to AuraDB Person nodes
# gds.louvain.write re-runs Louvain and writes writeProperty to the live database
write_result = gds.louvain.write(
    g,
    writeProperty='community',
    relationshipWeightProperty='weight',
)

print('Community IDs written back to Person nodes in AuraDB.')
print(f'  Nodes updated:     {write_result["nodePropertiesWritten"]}')
print(f'  Communities found: {write_result["communityCount"]}')
print(f'  Modularity:        {write_result["modularity"]:.4f}')

In [ ]:
# Verify: sample Person nodes now have .community set
verify = gds.run_cypher("""
MATCH (p:Person)
WHERE p.community IS NOT NULL
WITH count(p) AS personsWithCommunity
MATCH (p2:Person)
RETURN personsWithCommunity, count(p2) AS totalPersons
""")
print('Verification:')
print(verify.to_string(index=False))

In [ ]:
# Top communities by size (from live AuraDB data)
top_communities = gds.run_cypher("""
MATCH (p:Person)
WHERE p.community IS NOT NULL
WITH p.community AS community, collect(p.login) AS members, count(p) AS size
ORDER BY size DESC
LIMIT 10
RETURN community, size, members[0..5] AS sample_members
""")
print('Top 10 communities by size (from AuraDB):')
top_communities

## 7. Cleanup

Drop the in-memory GDS graph projection to free memory. The `Person.community` property persists in AuraDB.

In [ ]:
# Drop the in-memory projection to free server memory
# Person.community values already persisted to AuraDB in the write step above
gds.graph.drop(g)
print(f'Dropped in-memory projection: {GRAPH_NAME}')
print('Person.community values persist in AuraDB — ready for task-012 (PageRank + neo4j-viz).')

In [ ]:
# Close the GDS connection
gds.close()
print('Connection closed.')